# 03 — Modeling scratchpad

Shared experiments. Production code stays in `src/`.

In [2]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Cannot find repository root from current working directory.")


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix


In [3]:
PROCESSED_DIR = ROOT / "data" / "processed"

ratings_cf = pd.read_parquet(PROCESSED_DIR / "ratings_cf.parquet")
ratings_content = pd.read_parquet(PROCESSED_DIR / "ratings_content.parquet")
movies = pd.read_parquet(PROCESSED_DIR / "movies_clean.parquet")


In [4]:
ratings_cf.head(5)
ratings_cf.shape

(24639412, 4)

In [5]:
ratings_content.head(5)
ratings_content.shape

(24945390, 4)

In [6]:
movies.head(5)

,movieId,title,genres,year,genres_list,genres_text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,"[Adventure, Animation, Children, Comedy, Fantasy]",Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995,"[Adventure, Children, Fantasy]",Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,1995,"[Comedy, Romance]",Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,"[Comedy, Drama, Romance]",Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,1995,[Comedy],Comedy


In [7]:
def train_test_Split(
        ratings : pd.DataFrame,
        test_radio : 0.2,
        min_test : int = 1
):
    colums = {"userId", "movieId", "rating", "timestamp"}
    missing_columns = colums - set(ratings.columns)
    if missing_columns:
        raise ValueError(f"Thiếu các cột : {missing_columns}")
    if not 0<test_radio<1:
        raise ValueError("test_radio nằm trong khoảng (0,1)")
    data = ratings.copy()
    data.sort_values(
        by=["userId", "timestamp"]
    )
    data["positon"] = data.groupby("userId").cumcount()
    data["user_rating_count"] = data.groupby("userId")["movieId"].transform("size")
    data["n_test"] = np.maximum(
        np.floor(data["user_rating_count"] * test_radio).astype(int),
        min_test
    )
    data["n_test"] = np.minimum(
        data["n_test"],
        data["user_rating_count"] - 1
    )
    test_mask = (
        data["positon"] >= data["user_rating_count"] - data["n_test"]
    )
    train = data.loc[~test_mask].copy()
    test = data.loc[test_mask].copy()
    helper_columns = [
        "positon",
        "user_rating_count",
        "n_test"
    ]
    train.drop(columns=helper_columns, inplace=True)
    test.drop(columns=helper_columns, inplace=True)
    train.reset_index(drop=True, inplace=True)
    test.reset_index(drop=True, inplace=True)
    return train, test

In [8]:
ratings_content.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [9]:
ratings_cf.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [10]:
train_rating_cf, test_rating_cf = train_test_Split(ratings_cf, 0.2)

In [11]:
is_all_included = ratings_cf["movieId"].isin(movies["movieId"]).all()

print(is_all_included)

True


In [12]:
train_movies_cf = movies[movies["movieId"].isin(train_rating_cf["movieId"])]
test_movies_cf = movies[movies["movieId"].isin(test_rating_cf["movieId"])]

In [13]:
test_movies_cf.head(5)

,movieId,title,genres,year,genres_list,genres_text
69,70,From Dusk Till Dawn (1996),Action|Comedy|Horror|Thriller,1996,"[Action, Comedy, Horror, Thriller]",Action Comedy Horror Thriller
70,71,Fair Game (1995),Action,1995,[Action],Action
73,74,Bed of Roses (1996),Drama|Romance,1996,"[Drama, Romance]",Drama Romance
74,75,Big Bully (1996),Comedy|Drama,1996,"[Comedy, Drama]",Comedy Drama
75,76,Screamers (1995),Action|Sci-Fi|Thriller,1995,"[Action, Sci-Fi, Thriller]",Action Sci-Fi Thriller


In [14]:
movie_code , movie_ids = pd.factorize(
    train_rating_cf["movieId"],
    sort=True
)
user_code, user_id = pd.factorize(
    train_rating_cf["userId"],
    sort=True
)

In [15]:
from scipy.sparse import csr_matrix

movie_user_matrix = csr_matrix(
    (
        train_rating_cf["rating"],
        (movie_code, user_code)
    ),
    shape=(len(movie_ids), len(user_id))
)
print(movie_user_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 19773878 stored elements and shape (12948, 162242)>
  Coords	Values
  (0, 1)	3.5
  (0, 2)	4.0
  (0, 3)	3.0
  (0, 4)	4.0
  (0, 7)	4.0
  (0, 9)	3.5
  (0, 11)	4.0
  (0, 12)	4.0
  (0, 17)	3.0
  (0, 25)	3.0
  (0, 35)	5.0
  (0, 42)	4.0
  (0, 46)	2.0
  (0, 49)	4.0
  (0, 54)	2.0
  (0, 55)	4.0
  (0, 62)	4.0
  (0, 64)	3.0
  (0, 65)	3.0
  (0, 67)	3.0
  (0, 71)	3.0
  (0, 73)	5.0
  (0, 75)	4.0
  (0, 80)	4.0
  (0, 84)	5.0
  :	:
  (12928, 128838)	2.0
  (12929, 140860)	4.0
  (12930, 81757)	3.0
  (12930, 140860)	5.0
  (12931, 81757)	1.5
  (12932, 35998)	4.0
  (12932, 81757)	2.5
  (12933, 81757)	1.5
  (12934, 81757)	2.0
  (12935, 81757)	0.5
  (12936, 4076)	3.5
  (12937, 35998)	3.0
  (12937, 81757)	3.5
  (12938, 81757)	1.0
  (12939, 81757)	2.5
  (12940, 159848)	3.0
  (12941, 96985)	4.0
  (12942, 81757)	1.0
  (12942, 96985)	4.5
  (12943, 81757)	2.5
  (12944, 35998)	4.0
  (12944, 81757)	2.5
  (12945, 35998)	4.5
  (12946, 35998)	1.5
  (12947, 969

In [16]:
knn_cf = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=20
)

knn_cf.fit(movie_user_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",20
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [17]:
def recommend(movie_id, top_k=10):
    if movie_id not in movie_ids:
        print("Movie không tồn tại trong tập train")
        return None
    movie_index = movie_ids.get_loc(movie_id)
    movie_vector = movie_user_matrix[movie_index]
    distances, indices = knn_cf.kneighbors(
        movie_vector,
        n_neighbors=top_k + 1
    )

    distances = distances.flatten()
    indices = indices.flatten()

    results = []

    for distance, index in zip(distances, indices):
        recommended_movie_id = movie_ids[index]
        if recommended_movie_id == movie_id:
            continue
        results.append({
            "movieId": recommended_movie_id,
            "similarity": 1 - distance
        })
        if len(results) == top_k:
            break
    result_df = pd.DataFrame(results)
    result_df = result_df.merge(
        train_movies_cf[["movieId", "title"]],
        on="movieId",
        how="left"
    )
    return result_df[["movieId", "title", "similarity"]]

In [18]:
recommend(5, 10)

,movieId,title,similarity
0,3,Grumpier Old Men (1995),0.440504
1,62,Mr. Holland's Opus (1995),0.404445
2,7,Sabrina (1995),0.394784
3,141,"Birdcage, The (1996)",0.366464
4,95,Broken Arrow (1996),0.343978
5,736,Twister (1996),0.342072
6,376,"River Wild, The (1994)",0.336500
7,494,Executive Decision (1996),0.335723
8,708,"Truth About Cats & Dogs, The (1996)",0.317675
9,140,Up Close and Personal (1996),0.307986


In [19]:
users_like_5 = train_rating_cf[
    (train_rating_cf["movieId"] == 5) &
    (train_rating_cf["rating"] >= 4)
]["userId"]

In [20]:
other_movies = train_rating_cf[
    (train_rating_cf["userId"].isin(users_like_5)) &
    (train_rating_cf["rating"] >= 4)
]

In [21]:
other_movies["movieId"].value_counts().head(20)

movieId
5      3159
356    1513
1      1511
62     1271
150    1234
318    1213
457    1184
260    1123
480    1113
364    1103
500    1089
110    1060
339    1013
780     991
377     973
733     938
141     932
7       883
380     876
3       871
Name: count, dtype: int64

In [22]:
def recommend_for_user(user_id, top_k=10):
    user_history = train_rating_cf[
        train_rating_cf["userId"] == user_id
    ]

    user_like = user_history[
        user_history["rating"] >= 4
    ]["movieId"].tolist()

    if len(user_like) == 0:
        return False

    scores = {}

    # Lấy phim tương tự cho từng phim user thích
    for movie_id in user_like:
        neighbors = recommend(movie_id, top_k=10)

        for _, row in neighbors.iterrows():
            neighbor_movie = int(row["movieId"])
            similarity = float(row["similarity"])

            scores[neighbor_movie] = (
                scores.get(neighbor_movie, 0)
                + similarity
            )

    # Loại các phim user đã xem
    seen_movies = set(user_history["movieId"])

    for movie_id in seen_movies:
        scores.pop(movie_id, None)

    # Sắp xếp giảm dần theo tổng similarity
    sorted_scores = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    top_results = sorted_scores[:top_k]

    top_movie_ids = [
        movie_id
        for movie_id, score in top_results
    ]

    score_map = dict(top_results)

    result = movies[
        movies["movieId"].isin(top_movie_ids)
    ].copy()

    result["score"] = result["movieId"].map(score_map)

    # Giữ đúng thứ tự điểm giảm dần
    result = result.sort_values(
        "score",
        ascending=False
    )

    return result.reset_index(drop=True)

In [23]:
recommend_for_user(
    user_id=154472,
    top_k=10
)

,movieId,title,genres,year,genres_list,genres_text,score
0,141,"Birdcage, The (1996)",Comedy,1996,[Comedy],Comedy,2.702008
1,377,Speed (1994),Action|Romance|Thriller,1994,"[Action, Romance, Thriller]",Action Romance Thriller,2.549344
2,608,Fargo (1996),Comedy|Crime|Drama|Thriller,1996,"[Comedy, Crime, Drama, Thriller]",Comedy Crime Drama Thriller,2.466514
3,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,1994,"[Comedy, Crime, Drama, Thriller]",Comedy Crime Drama Thriller,2.278263
4,357,Four Weddings and a Funeral (1994),Comedy|Romance,1994,"[Comedy, Romance]",Comedy Romance,2.137316
5,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,1993,"[Action, Adventure, Sci-Fi, Thriller]",Action Adventure Sci-Fi Thriller,1.743793
6,1089,Reservoir Dogs (1992),Crime|Mystery|Thriller,1992,"[Crime, Mystery, Thriller]",Crime Mystery Thriller,1.710611
7,150,Apollo 13 (1995),Adventure|Drama|IMAX,1995,"[Adventure, Drama, IMAX]",Adventure Drama IMAX,1.642342
8,356,Forrest Gump (1994),Comedy|Drama|Romance|War,1994,"[Comedy, Drama, Romance, War]",Comedy Drama Romance War,1.637237
9,95,Broken Arrow (1996),Action|Adventure|Thriller,1996,"[Action, Adventure, Thriller]",Action Adventure Thriller,1.533961


In [24]:
test_positive = test_rating_cf[
    test_rating_cf["rating"] >= 4
]

In [25]:
sample_users = (
    test_positive["userId"]
    .drop_duplicates()
    .head(1)
)

precisions = []
recalls = []
hits = []

for user_id in sample_users:
    truth = set(
        test_positive[
            test_positive["userId"] == user_id
        ]["movieId"]
    )

    pred = recommend_for_user(
        user_id=user_id,
        top_k=10
    )

    if pred is False or len(pred) == 0:
        continue

    predicted = set(pred["movieId"])
    matched = predicted & truth

    precisions.append(len(matched) / 10)
    recalls.append(len(matched) / len(truth))
    hits.append(int(len(matched) > 0))

print("Precision@10:", np.mean(precisions))
print("Recall@10:", np.mean(recalls))
print("HitRate@10:", np.mean(hits))
print("Số user đã đánh giá:", len(precisions))

Precision@10: 0.0
Recall@10: 0.0
HitRate@10: 0.0
Số user đã đánh giá: 1


content


In [63]:
movies = pd.read_parquet(PROCESSED_DIR / "movies_clean.parquet")
movies.columns

Index(['movieId', 'title', 'genres', 'year', 'genres_list', 'genres_text'], dtype='str')

In [75]:
movies = movies.copy()
movies = movies.reset_index(drop=True)

movies["title"] = movies["title"].fillna("")
movies["genres_text"] = movies["genres_text"].fillna("")

movies["year"] = (
    movies["year"]
    .astype("string")
    .fillna("")
)

movies["clean_title"] = (
    movies["title"]
    .str.replace(r"\(\d{4}\)", "", regex=True)
    .str.lower()
    .str.strip()
)

movies["genres_text"] = (
    movies["genres_text"]
    .str.replace("|", " ", regex=False)
    .str.lower()
)

movies["feature_text"] = (
    movies["clean_title"]
    + " "
    + movies["genres_text"]
)

In [76]:
print(movies["year"].dtype)

movies[
    [
        "movieId",
        "title",
        "genres_text",
        "year",
        "feature_text"
    ]
].head()

string


,movieId,title,genres_text,year,feature_text
0,1,Toy Story (1995),adventure animation children comedy fantasy,1995,toy story adventure animation children comedy ...
1,2,Jumanji (1995),adventure children fantasy,1995,jumanji adventure children fantasy
2,3,Grumpier Old Men (1995),comedy romance,1995,grumpier old men comedy romance
3,4,Waiting to Exhale (1995),comedy drama romance,1995,waiting to exhale comedy drama romance
4,5,Father of the Bride Part II (1995),comedy,1995,father of the bride part ii comedy


In [79]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# Tạo ma trận đặc trưng TF-IDF
tfidf = TfidfVectorizer(
    stop_words="english",
    min_df=2,
    max_features=30000,
    ngram_range=(1, 2)
)

movie_feature_matrix = tfidf.fit_transform(
    movies["feature_text"]
)
# Tạo mapping
movie_id_to_index = pd.Series(
    movies.index,
    index=movies["movieId"]
).to_dict()

index_to_movie_id = pd.Series(
    movies["movieId"].values,
    index=movies.index
).to_dict()
print("TF-IDF matrix:", movie_feature_matrix.shape)

# Huấn luyện KNN trên ma trận TF-IDF
knn_content = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=21,
    n_jobs=-1
)

knn_content.fit(movie_feature_matrix)

print("KNN Content-Based đã fit xong")

TF-IDF matrix: (62423, 26311)
KNN Content-Based đã fit xong


In [80]:
def recommend_similar_movies(movie_id, top_k=10):
    movie_id = int(movie_id)

    if movie_id not in movie_id_to_index:
        return pd.DataFrame(
            columns=[
                "movieId",
                "title",
                "genres",
                "content_score"
            ]
        )

    movie_index = movie_id_to_index[movie_id]

    n_neighbors = min(top_k + 1, len(movies))

    distances, indices = knn_content.kneighbors(
        movie_feature_matrix[movie_index],
        n_neighbors=n_neighbors
    )

    results = []

    for distance, index in zip(
        distances.flatten(),
        indices.flatten()
    ):
        recommended_id = int(index_to_movie_id[index])

        # Bỏ chính phim đầu vào
        if recommended_id == movie_id:
            continue

        results.append({
            "movieId": recommended_id,
            "title": movies.iloc[index]["title"],
            "genres": movies.iloc[index]["genres"],
            "content_score": 1 - float(distance)
        })

        if len(results) == top_k:
            break

    return pd.DataFrame(results)

In [81]:
recommend_similar_movies(
    movie_id=1,
    top_k=10
)

,movieId,title,genres,content_score
0,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.000000
1,201588,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,0.948802
2,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.894377
3,106022,Toy Story of Terror (2013),Animation|Children|Comedy,0.718479
4,204188,UglyDolls (2019),Adventure|Animation|Children|Comedy|Fantasy,0.615751
5,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,0.615751
6,114552,"Boxtrolls, The (2014)",Adventure|Animation|Children|Comedy|Fantasy,0.615751
7,115875,Toy Story Toons: Hawaiian Vacation (2011),Adventure|Animation|Children|Comedy|Fantasy,0.597745
8,115879,Toy Story Toons: Small Fry (2011),Adventure|Animation|Children|Comedy|Fantasy,0.594684
9,3400,We're Back! A Dinosaur's Story (1993),Adventure|Animation|Children|Fantasy,0.583170


In [82]:
movies[
    movies["title"].str.contains(
        "Matrix",
        case=False,
        na=False
    )
][["movieId", "title"]]

,movieId,title
2480,2571,"Matrix, The (1999)"
6247,6365,"Matrix Reloaded, The (2003)"
6809,6934,"Matrix Revolutions, The (2003)"
9284,27660,"Animatrix, The (2003)"
28793,132490,Return to Source: The Philosophy of The Matrix...
39665,157721,Armitage: Dual Matrix (2002)
46345,172255,The Matrix Revisited (2001)
49723,179489,The Living Matrix (2009)
50471,181103,Matrix of Evil (2003)


In [83]:
recommend_similar_movies(
    movie_id=2571,
    top_k=10
)

,movieId,title,genres,content_score
0,157721,Armitage: Dual Matrix (2002),Action|Adventure|Animation|Sci-Fi|Thriller,0.697382
1,4887,"One, The (2001)",Action|Sci-Fi|Thriller,0.610199
2,65552,Replicant (2001),Action|Sci-Fi|Thriller,0.610199
3,125145,BlackJacks (2014),Action|Sci-Fi|Thriller,0.610199
4,51412,Next (2007),Action|Sci-Fi|Thriller,0.610199
5,131190,Alienator (1990),Action|Sci-Fi|Thriller,0.610199
6,200264,Defective (2017),Action|Sci-Fi|Thriller,0.610199
7,200566,Firehead (1991),Action|Sci-Fi|Thriller,0.610199
8,7163,Paycheck (2003),Action|Sci-Fi|Thriller,0.610199
9,139807,Firequake (2014),Action|Sci-Fi|Thriller,0.610199
